In [ ]:
import rerun as rr
import numpy as np
spectraldata = np.load("RESULTS/SAMPLE2/spectral/spectral_pointcloud.npz")
print(spectraldata)

In [ ]:
xyz = spectraldata["xyz"]
# xyz: [N, 3]
spectra = spectraldata["spectra"]
# spectra: [N, B]
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer

# Impute NaNs with the mean of each spectral band
imputer = SimpleImputer(strategy="mean")
spectra_imputed = imputer.fit_transform(spectra)
# spectra_imputed: [N, B]

wavelengths = spectraldata["wavelengths"]
# wavelengths: [B]

# PCA decomposition in spectral space
pca = PCA(n_components=3, random_state=0)
spectra3pca = pca.fit_transform(spectra_imputed)
# spectra3pca: [N, 3]

# Turn PCA scores into displayable pseudo-RGB colors (robust to outliers)
lo_pca = np.percentile(spectra3pca, 1.0, axis=0, keepdims=True)
hi_pca = np.percentile(spectra3pca, 99.0, axis=0, keepdims=True)
den_pca = np.maximum(hi_pca - lo_pca, 1e-8)
spectra3pca_rgb = np.clip((spectra3pca - lo_pca) / den_pca, 0.0, 1.0)
spectra3pca_rgb = np.power(spectra3pca_rgb, 0.8)
# spectra3pca_rgb: [N, 3], float in [0, 1]

# False RGB from wavelength-aware integration on the visible range.
# We approximate color matching functions with smooth Gaussians and integrate
# over measured wavelengths instead of taking 3 nearest bands.
wl = wavelengths.astype(np.float32)  # [B]
visible = (wl >= 380.0) & (wl <= 780.0)  # [B]
wl_vis = wl[visible]  # [Bv]
spec_vis = spectra_imputed[:, visible]  # [N, Bv]

# Approximate CIE-like XYZ sensitivities (normalized Gaussian mixture)
x_bar = (
    1.056 * np.exp(-0.5 * ((wl_vis - 599.8) / 37.9) ** 2)
    + 0.362 * np.exp(-0.5 * ((wl_vis - 442.0) / 16.0) ** 2)
    - 0.065 * np.exp(-0.5 * ((wl_vis - 501.1) / 20.4) ** 2)
)
y_bar = (
    0.821 * np.exp(-0.5 * ((wl_vis - 568.8) / 46.9) ** 2)
    + 0.286 * np.exp(-0.5 * ((wl_vis - 530.9) / 16.3) ** 2)
)
z_bar = (
    1.217 * np.exp(-0.5 * ((wl_vis - 437.0) / 11.8) ** 2)
    + 0.681 * np.exp(-0.5 * ((wl_vis - 459.0) / 26.0) ** 2)
)
cmf = np.stack([x_bar, y_bar, z_bar], axis=1)  # [Bv, 3]
cmf = np.clip(cmf, 0.0, None)

# Trapezoidal wavelength integration weights
dwl = np.gradient(wl_vis)  # [Bv]
cmf_w = cmf * dwl[:, None]  # [Bv, 3]

# XYZ per point: [N, Bv] @ [Bv, 3] -> [N, 3]
xyz_color = spec_vis @ cmf_w

# XYZ -> linear sRGB
M_xyz_to_srgb = np.array(
    [
        [3.2406, -1.5372, -0.4986],
        [-0.9689, 1.8758, 0.0415],
        [0.0557, -0.2040, 1.0570],
    ],
    dtype=np.float32,
)
false_rgb_linear = xyz_color @ M_xyz_to_srgb.T  # [N, 3]
false_rgb_linear = np.clip(false_rgb_linear, 0.0, None)

# Robust display normalization
lo_false = np.percentile(false_rgb_linear, 1.0, axis=0, keepdims=True)
hi_false = np.percentile(false_rgb_linear, 99.0, axis=0, keepdims=True)
den_false = np.maximum(hi_false - lo_false, 1e-8)
false_rgb = np.clip((false_rgb_linear - lo_false) / den_false, 0.0, 1.0)

# sRGB gamma curve
false_rgb = np.where(
    false_rgb <= 0.0031308,
    12.92 * false_rgb,
    1.055 * np.power(false_rgb, 1.0 / 2.4) - 0.055,
)
false_rgb = np.clip(false_rgb, 0.0, 1.0)
# false_rgb: [N, 3]

# Per-wavelength grayscale stack (vectorized over all bands)
lo_band = np.percentile(spectra_imputed, 1.0, axis=0, keepdims=True)
hi_band = np.percentile(spectra_imputed, 99.0, axis=0, keepdims=True)
den_band = np.maximum(hi_band - lo_band, 1e-8)
gray_stack = np.clip((spectra_imputed - lo_band) / den_band, 0.0, 1.0)
gray_stack = np.power(gray_stack, 0.8)
# gray_stack: [N, B]

print(
    "xyz", xyz.shape,
    "spectra", spectra.shape,
    "false_rgb", false_rgb.shape,
    "gray_stack", gray_stack.shape,
    "pca_rgb", spectra3pca_rgb.shape,
    "wavelengths", wavelengths.shape,
)

In [ ]:
rr.init("Spectralcloud", spawn=False)
rr.save("RESULTS/SAMPLE2/spectral/spectral_cloud.rrd")
# Frame 0: false RGB
rr.set_time("view", sequence=0)
rr.log("pointcloud", rr.Points3D(xyz, colors=false_rgb))
rr.log("view/label", rr.TextDocument("frame 0: false RGB"))


In [ ]:

# Frames 1..B: one grayscale frame per wavelength band
for band_idx in range(gray_stack.shape[1]):
    rr.set_time("view", sequence=band_idx + 1)
    gray = gray_stack[:, band_idx:band_idx + 1]
    gray_rgb = np.repeat(gray, 3, axis=1)
    rr.log("pointcloud", rr.Points3D(xyz, colors=gray_rgb))
    rr.log("view/label", rr.TextDocument(f"frame {band_idx + 1}: {wavelengths[band_idx]:.1f} nm (grayscale)"))

# Final frame: PCA pseudo-RGB
final_frame = gray_stack.shape[1] + 1
rr.set_time("view", sequence=final_frame)
rr.log("pointcloud", rr.Points3D(xyz, colors=spectra3pca_rgb))
rr.log("view/label", rr.TextDocument(f"frame {final_frame}: PCA pseudo-RGB"))


In [ ]:

RR_ARATIO = 1080/1920
WIDTH = 1200
HEIGHT = WIDTH * RR_ARATIO
rr.notebook_show(width=int(WIDTH), height=int(HEIGHT))